# Multi-Signal LSTM Forecast Notebook

This notebook trains `price`, `load`, and `pv` LSTM forecasters.

Goals:
- keep one editable config block per signal
- show a clear epoch progress bar for each signal
- plot loss and prediction charts immediately after each signal finishes
- optionally create one combined weekly summary plot at the end


In [ ]:
from pathlib import Path
import sys
from pprint import pprint

import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root


In [ ]:
from common.torch_runtime import configure_torch_runtime, describe_device
from configs import compose_experiment_config
from forecast.artifacts import get_weekly_forecast_plot_path
from forecast.training import (
    collect_available_lstm_artifacts,
    plot_signal_training_report,
    plot_weekly_forecasts,
    train_signal_lstm,
)
        


## 1. Runtime and Per-Signal Settings

Keep shared settings and per-signal overrides separate so students can edit them quickly.


In [ ]:
runtime_mode = "performance"          # "performance" | "strict_reproducibility"
seed = 0
device_request = None                 # None -> auto select CUDA when available, else CPU
require_cuda = False

signal_order = ["price", "load", "pv"]
shared_settings = {
    "future_horizon": 24,
    "history_window": 96 * 7,
    "artifact_root": project_root / "artifacts" / "forecast" / "lstm",
}

signal_training_overrides = {
    "price": {
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.10,
        "batch_size": 1024,
        "epochs": 12,
        "lr": 1e-3,
    },
    "load": {
        "hidden_size": 96,
        "num_layers": 2,
        "dropout": 0.10,
        "batch_size": 1024,
        "epochs": 10,
        "lr": 1e-3,
    },
    "pv": {
        "hidden_size": 96,
        "num_layers": 1,
        "dropout": 0.00,
        "batch_size": 1024,
        "epochs": 10,
        "lr": 8e-4,
    },
}


In [ ]:
cfg = compose_experiment_config(
    observation_profile="simbench",
    forecast_type="lstm",
    data_dir=project_root / "data",
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)
cfg.data.dataset_type = "csv_prosumer"
cfg.obs.sequence_features = list(signal_order)
cfg.forecast.target_signals = list(signal_order)
cfg.env.future_horizon = int(shared_settings["future_horizon"])
cfg.forecast.history_window = int(shared_settings["history_window"])
cfg.forecast.lstm_artifact_root = Path(shared_settings["artifact_root"])

runtime_state = configure_torch_runtime(
    cfg,
    device=device_request,
    seed=seed,
    require_cuda=require_cuda,
)

experiment_summary = {
    "runtime_mode": cfg.runtime.execution_mode,
    "device": str(runtime_state.device),
    "device_info": describe_device(runtime_state),
    "future_horizon": cfg.env.future_horizon,
    "history_window": cfg.forecast.history_window,
    "artifact_root": str(cfg.forecast.lstm_artifact_root),
    "signal_order": list(signal_order),
    "signal_training_overrides": signal_training_overrides,
}
pprint(experiment_summary)
        


## 2. Helper for Train-Then-Plot

`train_signal_lstm(..., show_progress=True)` shows an epoch progress bar for the current signal.
The helper below trains one signal and renders its charts immediately.


In [ ]:
results = {}


def train_and_report_signal(signal_name: str):
    override = dict(signal_training_overrides[signal_name])
    effective = {
        "history_window": shared_settings["history_window"],
        "future_horizon": shared_settings["future_horizon"],
        **override,
    }

    print("=" * 88)
    print(f"[{signal_name}] training starts")
    pprint(effective)

    result = train_signal_lstm(
        cfg,
        signal_name,
        device=runtime_state,
        overrides=override,
        show_progress=True,
    )
    figure = plot_signal_training_report(result)
    plt.show()
    plt.close(figure)

    print(f"[{signal_name}] best_val_loss = {result['training']['best_val_loss']:.6f}")
    print(f"[{signal_name}] artifact_paths = {result['artifact_paths']}")
    return result


In [ ]:
results = {}
for signal_name in signal_order:
    results[signal_name] = train_and_report_signal(signal_name)
        


In [ ]:
plot_path = get_weekly_forecast_plot_path(
    future_horizon=cfg.env.future_horizon,
    root=cfg.forecast.lstm_artifact_root,
)
plot_weekly_forecasts(
    [results[signal_name]["evaluation"] for signal_name in signal_order],
    save_path=plot_path,
)

artifact_map = collect_available_lstm_artifacts(cfg)
final_summary = {
    signal_name: {
        "settings": results[signal_name]["settings"],
        "best_val_loss": results[signal_name]["training"]["best_val_loss"],
        "artifact_paths": results[signal_name]["artifact_paths"],
    }
    for signal_name in signal_order
}

print("combined weekly plot:", plot_path)
pprint(final_summary)
artifact_map


## 3. Full Test-Set View with Ideal Warmup

The cell below draws one-step rolling predictions for `price`, `load`, and `pv` with two constraints:
- split the test set by `segment_id`
- use per-user subplots for multi-user signals such as `load` and `pv`

It assumes an ideal warmup: each prediction uses the latest real `history_window` points from the same test segment.


In [ ]:
from forecast.lstm_forecaster import LSTMForecaster
from forecast.training import load_signal_matrix


def _build_signal_forecaster_from_result(signal_name: str):
    artifact_paths = results[signal_name]["artifact_paths"]
    return LSTMForecaster.from_artifacts(
        model_path=artifact_paths["model_path"],
        meta_path=artifact_paths["meta_path"],
        scaler_path=artifact_paths["scaler_path"],
        device=runtime_state.device,
        signal_name=signal_name,
    )


def _rolling_test_predictions_with_ideal_warmup(signal_name: str):
    source = results[signal_name]["source"]
    test_frame, test_values, value_columns = load_signal_matrix(source.test_path, signal_name)
    forecaster = _build_signal_forecaster_from_result(signal_name)
    history_window = int(results[signal_name]["settings"]["history_window"])

    values = np.asarray(test_values, dtype=np.float32)
    if values.ndim == 1:
        values = values.reshape(-1, 1)

    working_frame = test_frame.copy()
    if "segment_id" not in working_frame.columns:
        working_frame["segment_id"] = 0
    working_frame["segment_id"] = working_frame["segment_id"].fillna(0).astype(int)

    segment_views = []
    for segment_id in sorted(working_frame["segment_id"].unique()):
        segment_mask = working_frame["segment_id"] == segment_id
        segment_frame = working_frame.loc[segment_mask].reset_index(drop=True)
        segment_values = values[segment_mask.to_numpy()]

        if segment_values.shape[0] <= history_window:
            print(
                f"[skip] signal={signal_name}, segment_id={segment_id}: "
                f"rows={segment_values.shape[0]} <= history_window={history_window}"
            )
            continue

        timestamps = []
        target_rows = []
        prediction_rows = []

        for end_idx in range(history_window, segment_values.shape[0]):
            history = segment_values[end_idx - history_window:end_idx]
            rollout = forecaster.predict(history, horizon=2, signal_name=signal_name)
            rollout = np.asarray(rollout, dtype=np.float32)

            if rollout.ndim == 1:
                prediction_rows.append(np.array([rollout[1]], dtype=np.float32))
            else:
                prediction_rows.append(rollout[:, 1].astype(np.float32))

            target_rows.append(segment_values[end_idx].astype(np.float32))
            timestamps.append(segment_frame.iloc[end_idx]["timestamp"] if "timestamp" in segment_frame.columns else end_idx)

        segment_views.append(
            {
                "segment_id": int(segment_id),
                "timestamps": np.asarray(timestamps),
                "target": np.stack(target_rows, axis=0),
                "prediction": np.stack(prediction_rows, axis=0),
            }
        )

    if not segment_views:
        raise ValueError(
            f"signal={signal_name} has no test segment long enough for history_window={history_window}."
        )

    return {
        "signal_name": signal_name,
        "segments": segment_views,
        "value_columns": value_columns,
        "history_window": history_window,
    }


full_test_views = {
    signal_name: _rolling_test_predictions_with_ideal_warmup(signal_name)
    for signal_name in signal_order
}

for signal_name in signal_order:
    view = full_test_views[signal_name]
    n_users = len(view["value_columns"])
    figure, axes = plt.subplots(n_users, 1, figsize=(16, 3.6 * n_users), sharex=False)
    axes = np.atleast_1d(axes)

    for user_idx, axis in enumerate(axes):
        user_name = view["value_columns"][user_idx]
        for segment in view["segments"]:
            target_curve = segment["target"][:, user_idx]
            prediction_curve = segment["prediction"][:, user_idx]
            axis.plot(
                segment["timestamps"],
                target_curve,
                linewidth=1.5,
                label=f"Ground truth (segment {segment['segment_id']})",
            )
            axis.plot(
                segment["timestamps"],
                prediction_curve,
                linewidth=1.3,
                linestyle="--",
                label=f"Prediction (segment {segment['segment_id']})",
            )

        axis.set_title(
            f"{signal_name} - {user_name} rolling one-step forecast by segment "
            f"(ideal warmup={view['history_window']})"
        )
        axis.set_ylabel(user_name)
        axis.grid(True, alpha=0.3)
        axis.legend(loc="upper right", ncol=2)

    axes[-1].set_xlabel("timestamp")
    figure.tight_layout()
    plt.show()

full_test_views


## Smoke Check

Suggested minimal smoke check:
1. set `signal_order = ["price"]`
2. set `signal_training_overrides["price"]["epochs"] = 1`
3. run the training cell and confirm the `price epochs` progress bar appears
4. confirm the loss plot and prediction plot appear immediately after training
5. confirm the final cell lists artifacts under `artifacts/forecast/lstm/h{future_horizon}/price/`
